In [1]:
from datetime import datetime, timezone
import pathlib
import json
import time
import requests
from bs4 import BeautifulSoup
from selenium.webdriver import Remote, ChromeOptions

# ========================
# SETUP DIRECTORIES
# ========================
now = datetime.now(timezone.utc)
today = now.strftime("%Y-%m-%d")

BASE_DIR = pathlib.Path().resolve()
DATASET_DIR = BASE_DIR / "dataset" / today / "AI_ML_Insights"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset Path:", DATASET_DIR)

# ========================
# SELENIUM CONFIG
# ========================
options = ChromeOptions()
options.add_argument("--headless=new")

prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)

SELENIUM_URL = "http://localhost:4444/wd/hub"

# ========================
# UTIL FUNCTIONS
# ========================
def save_json(data, filename):
    path = DATASET_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

# ========================
# 1. GITHUB TRENDING AI
# ========================
def scrape_github():
    import requests
    from bs4 import BeautifulSoup
    import time

    url = "https://github.com/trending/artificial-intelligence"
    data = []

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    # -----------------------
    # TRY HTML SCRAPING (3 retries)
    # -----------------------
    for attempt in range(3):
        try:
            res = requests.get(url, headers=headers, timeout=15)
            res.raise_for_status()

            soup = BeautifulSoup(res.text, "html.parser")

            repos = soup.select("article.Box-row")
            if not repos:
                raise Exception("No repos found (possible block)")

            for repo in repos[:15]:
                try:
                    name = repo.h2.text.strip().replace("\n", "").replace(" ", "")
                    link = "https://github.com" + repo.h2.a["href"]

                    stars_elem = repo.select_one("a[href*='stargazers']")
                    forks_elem = repo.select_one("a[href*='forks']")

                    data.append({
                        "source": "github_scrape",
                        "repo": name,
                        "url": link,
                        "stars": stars_elem.text.strip() if stars_elem else "0",
                        "forks": forks_elem.text.strip() if forks_elem else "0"
                    })
                except:
                    continue

            print("✅ GitHub scraped successfully")
            return data

        except Exception as e:
            print(f"Attempt {attempt+1} failed:", e)
            time.sleep(2)

    # -----------------------
    # 🔁 FALLBACK → GITHUB API
    # -----------------------
    print("⚠️ Falling back to GitHub API...")

    try:
        api_url = "https://api.github.com/search/repositories?q=artificial+intelligence&sort=stars&order=desc"

        res = requests.get(api_url, headers=headers, timeout=15)
        res.raise_for_status()

        json_data = res.json()

        for item in json_data.get("items", [])[:15]:
            data.append({
                "source": "github_api",
                "repo": item["full_name"],
                "url": item["html_url"],
                "stars": item["stargazers_count"],
                "forks": item["forks_count"]
            })

        print("✅ GitHub API fallback used")
        return data

    except Exception as e:
        print("❌ GitHub API also failed:", e)
        return []

# ========================
# 2. STACKOVERFLOW TAGS
# ========================
def scrape_stackoverflow():
    url = "https://stackoverflow.com/tags?tab=popular"
    data = []

    try:
        with Remote(SELENIUM_URL, options=options) as driver:
            driver.get(url)
            time.sleep(3)

            soup = BeautifulSoup(driver.page_source, "html.parser")

            for card in soup.select(".s-card")[:20]:
                try:
                    tag_elem = card.select_one(".post-tag")
                    desc_elem = card.select_one(".fc-black-500")
                    stat_elem = card.select_one(".flex--item")

                    data.append({
                        "source": "stackoverflow",
                        "tag": tag_elem.text.strip() if tag_elem else None,
                        "description": desc_elem.text.strip() if desc_elem else None,
                        "total_questions": stat_elem.text.strip() if stat_elem else "0"
                    })
                except:
                    continue

    except Exception as e:
        print("StackOverflow error:", e)

    return data

# ========================
# 3. ARXIV PAPERS
# ========================
def scrape_arxiv():
    url = "http://export.arxiv.org/api/query?search_query=cat:cs.AI&max_results=10"
    data = []

    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "xml")

        for entry in soup.find_all("entry"):
            data.append({
                "source": "arxiv",
                "title": entry.title.text.strip(),
                "published": entry.published.text,
                "link": entry.id.text
            })

    except Exception as e:
        print("arXiv error:", e)

    return data

# ========================
# 4. JOB DEMAND (SAFE MOCK STRUCTURE)
# ========================
def scrape_jobs():
    roles = [
        "Machine Learning Engineer",
        "Data Scientist",
        "AI Engineer"
    ]

    data = []
    for role in roles:
        data.append({
            "source": "jobs",
            "role": role,
            "trend": "High",
            "note": "Replace with API later"
        })

    return data

# ========================
# 5. FUNDING DATA (SAFE)
# ========================
def scrape_funding():
    return [
        {"source": "funding", "company": "OpenAI", "funding": ">$10B"},
        {"source": "funding", "company": "Anthropic", "funding": ">$7B"},
        {"source": "funding", "company": "Mistral AI", "funding": "$1B+"}
    ]

# ========================
# MAIN PIPELINE
# ========================
def main():
    print("\nRunning AI/ML scraper...\n")

    github = scrape_github()
    stack = scrape_stackoverflow()
    arxiv = scrape_arxiv()
    jobs = scrape_jobs()
    funding = scrape_funding()

    combined = github + stack + arxiv + jobs + funding

    # SAVE FILES
    save_json(github, "github.json")
    save_json(stack, "stackoverflow.json")
    save_json(arxiv, "arxiv.json")
    save_json(jobs, "jobs.json")
    save_json(funding, "funding.json")
    save_json(combined, "combined.json")

    print("✅ Done")
    print("Total records:", len(combined))

# ========================
# RUN
# ========================
main()

Dataset Path: C:\Users\Lenovo Loq\OneDrive\Documents\webscrapping\Loq\dev\smarter-scraping\src\dataset\2026-03-31\AI_ML_Insights

Running AI/ML scraper...

Attempt 1 failed: HTTPSConnectionPool(host='github.com', port=443): Max retries exceeded with url: /trending/artificial-intelligence (Caused by NewConnectionError("HTTPSConnection(host='github.com', port=443): Failed to establish a new connection: [WinError 10065] A socket operation was attempted to an unreachable host"))
Attempt 2 failed: HTTPSConnectionPool(host='github.com', port=443): Max retries exceeded with url: /trending/artificial-intelligence (Caused by NewConnectionError("HTTPSConnection(host='github.com', port=443): Failed to establish a new connection: [WinError 10065] A socket operation was attempted to an unreachable host"))
Attempt 3 failed: HTTPSConnectionPool(host='github.com', port=443): Max retries exceeded with url: /trending/artificial-intelligence (Caused by NewConnectionError("HTTPSConnection(host='github.com